In [ ]:
""" 
Proposed by Jules Nde, PhD, Washburn Lab, Cancer Biology, University of Kansas Medical Center (25/8/2025)

Author: Jules Nde, Washburn Lab, Cancer Biology, University of Kansas Medical Center
Copyright © 2025 by Jules Nde. All rights reserved. No part of this work may be reproduced or transmitted in any form without permission.

This script combines multiple models and chains from a given protein complex into one chain, then renumberes residues starting at 1.
    
Parameters:
    - input_pdb (str): Path to the input PDB file.
    - output_pdb (str): Path to the output PDB file with renumbered residues.
"""


from Bio.PDB import PDBParser, PDBIO, StructureBuilder
from Bio.PDB.Residue import Residue
from Bio.PDB.Atom import Atom
import os

def combine_models_to_single_chain(input_pdb, output_pdb, chain_id='A'):
    """
    Combines all models and chains in a PDB into one chain,
    renumbering residues from 1 and writing a new PDB file.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("combined", input_pdb)

    builder = StructureBuilder.StructureBuilder()
    builder.init_structure("combined")
    builder.init_model(0)
    builder.init_chain(chain_id)

    residue_id = 1  # Start renumbering from 1

    for model in structure:
        for chain in model:
            for residue in chain:
                hetfield, resseq, icode = residue.get_id()
                if hetfield.strip() != '' or residue.get_resname() == 'HOH':
                    continue  # Skip water and heteroatoms

                new_residue = Residue((' ', residue_id, ' '), residue.get_resname(), residue.get_segid())

                for atom in residue:
                    new_atom = Atom(atom.get_name(),
                                    atom.get_coord(),
                                    atom.get_bfactor(),
                                    atom.get_occupancy(),
                                    atom.get_altloc(),
                                    atom.get_fullname(),
                                    atom.get_serial_number(),
                                    element=atom.element)
                    new_residue.add(new_atom)

                builder.structure[0][chain_id].add(new_residue)
                residue_id += 1

    io = PDBIO()
    io.set_structure(builder.get_structure())
    io.save(output_pdb)
    print(f"✅ Combined structure saved to: {output_pdb}")

# Example usage (update the filenames)
combine_models_to_single_chain("Rpd3L_incompl_recost_9_24_25_w_1Ume1.pdb", "Rpd3L_incompl_recost_9_24_25_w_1Ume1_chain.pdb") #(input.pdb, output.pdb)
